In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm
from google import genai
from google.genai import types
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

PROJECT_ROOT = Path.cwd()
with open(PROJECT_ROOT / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

paths = config["paths"]
bertopic_config = config["bertopic"]
gemini_config = config["gemini"]
topics_over_time_config = config["topics_over_time"]

: 

In [7]:
# =====================================================================
# 1. SETUP API KEY AND GEMINI CLIENT
# =====================================================================
if not os.environ.get("GEMINI_API_KEY"):
    env_path = PROJECT_ROOT / ".env"
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if line.startswith("GEMINI_API_KEY="):
                os.environ["GEMINI_API_KEY"] = line.split("=", 1)[1].strip()
                break

# Initialize the new standard Google GenAI client
client = genai.Client()

In [8]:
# =====================================================================
# 2. LOAD PREPROCESSED DATA & EMBEDDINGS
# =====================================================================
DATA_PATH = PROJECT_ROOT / paths["bertopic_ready"]
EMBEDDING_PATH = PROJECT_ROOT / paths["embeddings"]
LABELS_PATH = PROJECT_ROOT / paths["cluster_labels"]

df = pd.read_csv(DATA_PATH)

# --- AUTO-DETECT COLUMN NAME ---
# This step prevents KeyError by looking for 'clean_narrative' first,
# and falling back to 'Consumer complaint narrative' if not found.
if "clean_narrative" in df.columns:
    text_column = "clean_narrative"
elif "Consumer complaint narrative" in df.columns:
    text_column = "Consumer complaint narrative"
else:
    # If neither matches, default to the first column containing text
    text_column = df.select_dtypes(include=[object]).columns[0]

print(f"Using text column: '{text_column}' for topic modeling.")
documents = df[text_column].astype(str).tolist()

embeddings = np.load(EMBEDDING_PATH)
labels = np.load(LABELS_PATH)

print(f"Loaded {len(documents)} clean documents.")
print(f"Loaded embeddings matrix of shape: {embeddings.shape}")
print(f"Loaded custom labels array of shape: {labels.shape}")

Using text column: 'Consumer complaint narrative' for topic modeling.
Loaded 35993 clean documents.
Loaded embeddings matrix of shape: (35993, 2048)
Loaded custom labels array of shape: (35993,)


In [9]:
# =====================================================================
# 3. INITIALIZE & RUN BERTOPIC WITH OFFLINE LABELS (BYPASS MODE)
# =====================================================================
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.cluster import BaseCluster

# 1. Pre-configure the vectorizer to ignore custom/redacted noise words
vectorizer_model = CountVectorizer(stop_words=bertopic_config["vectorizer_stop_words"])

# 2. Tell BERTopic to skip ALL background training steps.
# We skip Embedding, UMAP, and HDBSCAN since we are feeding precomputed data.
topic_model = BERTopic(
    embedding_model=None,
    umap_model=BaseDimensionalityReduction(),
    hdbscan_model=BaseCluster(),
    vectorizer_model=vectorizer_model,
    calculate_probabilities=bertopic_config["calculate_probabilities"]
)

print("Calculating topic keywords (c-TF-IDF)... This will take under 15 seconds.")

# 3. Fit using your documents, custom embeddings, and offline labels
topics, _ = topic_model.fit_transform(
    documents,
    embeddings=embeddings,
    y=labels
)

# Extract generated topic configurations
topic_info = topic_model.get_topic_info()
print(f"\nDiscovered {len(topic_info) - 1} semantic clusters (excluding outliers).")

Calculating topic keywords (c-TF-IDF)... This will take under 15 seconds.



Discovered 213 semantic clusters (excluding outliers).


In [10]:
import json
import time
from tqdm import tqdm
from google.genai import types

# =====================================================================
# 4. GEMINI TOPIC NAMING PIPELINE (BATCH PROCESSING WITH JSON)
# =====================================================================
gemini_topic_labels = {}
gemini_topic_labels[-1] = bertopic_config["outlier_label"]

# 1. Filter out the outlier topic (-1)
valid_topics = topic_info[topic_info["Topic"] != -1]

# 2. Configure the Batch Size
BATCH_SIZE = gemini_config["topic_label_batch_size"]
batches = [valid_topics[i:i + BATCH_SIZE] for i in range(0, len(valid_topics), BATCH_SIZE)]

print(f"\nGrouped {len(valid_topics)} topics into {len(batches)} batches.")
print("Executing Gemini Batch Naming Pipeline...\n")

ACTIVE_MODEL = gemini_config["model"]

for batch_idx, batch in enumerate(tqdm(batches, desc="Processing Batches")):

    # Construct a structured prompt for the batch
    batch_data = []
    for _, row in batch.iterrows():
        t_id = row["Topic"]
        keywords = [word for word, _ in topic_model.get_topic(t_id)]
        rep_docs = row["Representative_Docs"]
        rep_doc_sample = (
            rep_docs[0][:gemini_config["representative_doc_chars"]]
            if rep_docs
            else "No sample text"
        )

        batch_data.append({
            "topic_id": int(t_id),
            "keywords": keywords[:gemini_config["label_keywords_limit"]],
            "sample_text": rep_doc_sample
        })

    prompt = f"""
You are an expert financial consumer complaint analyst. Your job is to generate highly concise, professional, and descriptive 3-to-5 word labels for multiple clusters of customer complaints.

Below is a JSON array of complaint clusters. For each object, analyze the 'keywords' and 'sample_text', then return a JSON array with the exact same topic_id and a concise label.

Complaint clusters:
{json.dumps(batch_data, indent=2)}

Return only JSON in this format:
[
  {{"topic_id": 0, "label": "Example Topic Label"}}
]
"""

    max_retries = gemini_config["max_retries"]
    retry_delay = gemini_config["retry_delay_seconds"]

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=ACTIVE_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                )
            )

            # Parse the structured JSON response
            batch_results = json.loads(response.text.strip())

            for item in batch_results:
                gemini_topic_labels[int(item["topic_id"])] = item["label"]

            break

        except Exception as e:
            print(f"\n[Error on Batch {batch_idx + 1}] {e}. Retrying in {retry_delay}s...")
            time.sleep(retry_delay)


Grouped 213 topics into 15 batches.
Executing Gemini Batch Naming Pipeline...



Processing Batches: 100%|Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†Ã¢â€“Ë†| 15/15 [01:23<00:00,  5.58s/it]


Batch labeling complete!


In [12]:
# =====================================================================
# 5. INTEGRATE LABELS AND SAVE
# =====================================================================
# Map the custom Gemini names to our local DataFrame
df["Topic"] = topics
df["Topic_Label"] = df["Topic"].map(gemini_topic_labels)

# Update the BERTopic model's internally tracked names
topic_model.set_topic_labels(gemini_topic_labels)

# Save the master analysis results
df.to_csv(PROJECT_ROOT / paths["final_labeled_topics"], index=False)
print(f"\nDone! Exported final labeled metrics to '{paths['final_labeled_topics']}'")

# Quick evaluation preview using the auto-detected text column variable
print(df[[text_column, "Topic", "Topic_Label"]].head(10))


Done! Exported final labeled metrics to 'cfpb_final_with_labeled_topics.csv'
                        Consumer complaint narrative  Topic  \
0  To Whom It May Concern, I am writing in respon...      1   
1  on Monday, [PROTECTED]/[PROTECTED]/[PROTECTED]...     11   
2  I am writing to bring to your attention an iss...     -1   
3  some people got hold of debit card information...     15   
4  Good day! I am reaching out regarding several ...     37   
5  It has been drawn out into the open that you a...     -1   
6  I am extremely dissatisfied with the failure t...    189   
7  I am writing to formally request the removal o...    157   
8  I am initiating a dispute and investigation re...    194   
9  [PROTECTED] [PROTECTED] [PROTECTED] opened an ...    200   

                                 Topic_Label  
0            Formal Debt Validation Requests  
1  Unauthorized Digital Payment Transactions  
2                    Outliers / Unclassified  
3           Credit Card Transaction Disp

In [13]:
# Save the fitted BERTopic model safely to disk
topic_model.save(
    PROJECT_ROOT / paths["bertopic_model_dir"],
    serialization=bertopic_config["serialization"]
)
print("Model saved successfully!")

Model saved successfully!


In [14]:
# 1. Interactive Intertopic Distance Map (Shows how topics relate to each other)
fig_topics = topic_model.visualize_topics(custom_labels=True)
fig_topics.show()

# 2. Visualize Topic Hierarchy (Shows how topics naturally merge together)
fig_hierarchy = topic_model.visualize_hierarchy(custom_labels=True)
fig_hierarchy.show()

# 3. Save these interactive charts as HTML files to share with others!
fig_topics.write_html(PROJECT_ROOT / paths["intertopic_distance_map_html"])
fig_hierarchy.write_html(PROJECT_ROOT / paths["topic_hierarchy_html"])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [15]:
# If your original CSV has a date column (e.g., "Date received")
if "Date received" in df.columns:
    df["Date received"] = pd.to_datetime(df["Date received"])

    # Calculate how topics change over time
    topics_over_time = topic_model.topics_over_time(
        docs=documents,
        timestamps=df["Date received"].tolist()
    )

    # Plot it!
    fig_time = topic_model.visualize_topics_over_time(topics_over_time, custom_labels=True)
    fig_time.show()
    fig_time.write_html(PROJECT_ROOT / paths["topics_evolution_over_time_html"])

2026-07-16 17:55:38,458 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 35963) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 


KeyboardInterrupt: 

In [18]:
# Group and count topics
top_complaints = df["Topic_Label"].value_counts().head(100)

print("Ã°Å¸Ââ€  TOP 10 MOST COMMON CONSUMER COMPLAINTS:")
print("-" * 50)
for label, count in top_complaints.items():
    percentage = (count / len(df)) * 100
    print(f"Ã°Å¸â€Â¹ {label:<45} | Count: {count:<6} ({percentage:.2f}%)")

Ã°Å¸Ââ€  TOP 10 MOST COMMON CONSUMER COMPLAINTS:
--------------------------------------------------
Ã°Å¸â€Â¹ Outliers / Unclassified                       | Count: 11545  (32.08%)
Ã°Å¸â€Â¹ Identity Theft Fraud Reporting                | Count: 1648   (4.58%)
Ã°Å¸â€Â¹ Formal Debt Validation Requests               | Count: 967    (2.69%)
Ã°Å¸â€Â¹ Credit Report Accuracy Disputes               | Count: 914    (2.54%)
Ã°Å¸â€Â¹ Mortgage Escrow Payment Issues                | Count: 772    (2.14%)
Ã°Å¸â€Â¹ Unauthorized Hard Inquiry Disputes            | Count: 546    (1.52%)
Ã°Å¸â€Â¹ Unauthorized Credit Inquiry Removal           | Count: 505    (1.40%)
Ã°Å¸â€Â¹ Credit Card Fee Disagreements                 | Count: 478    (1.33%)
Ã°Å¸â€Â¹ Auto Loan Repossession Disputes               | Count: 426    (1.18%)
Ã°Å¸â€Â¹ Consumer Privacy Rights Violation             | Count: 371    (1.03%)
Ã°Å¸â€Â¹ Cash App Fraud Complaints                     | Count: 308    (0.86%)
Ã°Å¸â€Â¹ Credit V

In [17]:
import pandas as pd

# =====================================================================
# 1. PREPARE DATETIME STRINGS
# =====================================================================
# Auto-detect the date column from your CSV (usually "Date received" in CFPB)
date_column = None
for col in ["Date received", "date_received", "date", "Date"]:
    if col in df.columns:
        date_column = col
        break

if date_column is None:
    raise ValueError("Could not find a date column in your CSV. Please ensure you have a date receipt column.")

print(f"Using date column: '{date_column}'")

# Convert the dates column into standard pandas datetime objects
df[date_column] = pd.to_datetime(df[date_column])
timestamps = df[date_column].tolist()

# =====================================================================
# 2. CALCULATE TOPIC EVOLUTION (DYNAMIC TOPIC MODELING)
# =====================================================================
print("Calculating topic distributions over time...")
# This generates the frequencies of each topic across temporal bins
topics_over_time = topic_model.topics_over_time(
    docs=documents,
    timestamps=timestamps,
    nr_bins=topics_over_time_config["nr_bins"]
)

# =====================================================================
# 3. CHOOSE A SPECIFIC TOPIC TO PROVE THE CONCEPT
# =====================================================================
TARGET_TOPICS = topics_over_time_config["target_topics"]

print(f"Generating interactive timeline visualization for topics: {TARGET_TOPICS}")

# Generate the plot strictly for our selected target topics
fig_timeline = topic_model.visualize_topics_over_time(
    topics_over_time,
    topics=TARGET_TOPICS,
    custom_labels=True
)

# Save it as an interactive, draggable HTML page
fig_timeline.write_html(PROJECT_ROOT / paths["specific_topics_over_time_html"])
print(f"Done! Open '{paths['specific_topics_over_time_html']}' in your browser to see your interactive trend lines.")

# Show it directly in the notebook if your environment is fully resolved
fig_timeline.show()

Using date column: 'Date received'
Calculating topic distributions over time...
Generating interactive timeline visualization for topics: [0, 1]
Done! Open 'specific_topics_over_time.html' in your browser to see your interactive trend lines.


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
import webbrowser

# Get the absolute path of the HTML file
file_path = (PROJECT_ROOT / paths["specific_topics_over_time_html"]).resolve()

# Open it in your default web browser (Chrome, Edge, Safari, Firefox, etc.)

webbrowser.open(f"file://{file_path}")

True

In [3]:
import pandas as pd

df=pd.read_csv("cfpb_final_with_labeled_topics.csv")

In [4]:
df

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,word_count,language,Topic,Topic_Label
0,2025-09-18T02:30:29.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"To Whom It May Concern, I am writing in respon...",NaN,"EQUIFAX, INC.",NY,109XX,NaN,Web,2025-09-18T02:30:55.000Z,Closed with explanation,Yes,16011810.0,439,Language.ENGLISH,1,Formal Debt Validation Requests
1,2026-03-13T00:27:38.000Z,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Transaction was not authorized,"on Monday, [PROTECTED]/[PROTECTED]/[PROTECTED]...",NaN,"Block, Inc.",OR,97317,NaN,Web,2026-03-13T00:46:10.000Z,Closed with explanation,Yes,20223337.0,144,Language.ENGLISH,11,Unauthorized Digital Payment Transactions
2,2025-05-23T22:58:46.000Z,"Payday loan, title loan, personal loan, or adv...",Payday loan,Money was taken from your bank account on the ...,NaN,I am writing to bring to your attention an iss...,Company believes the complaint provided an opp...,Albert Corporation,CT,06405,NaN,Web,2025-05-23T23:27:55.000Z,Closed with explanation,Yes,13691637.0,266,Language.ENGLISH,-1,Outliers / Unclassified
3,2025-07-26T23:50:05.000Z,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Transaction was not authorized,some people got hold of debit card information...,NaN,"HUNTINGTON NATIONAL BANK, THE",MN,554XX,Older American,Web,2025-07-27T00:49:31.000Z,Closed with monetary relief,Yes,14903535.0,99,Language.ENGLISH,15,Credit Card Transaction Disputes
4,2024-12-26T20:40:18.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Good day! I am reaching out regarding several ...,NaN,"EQUIFAX, INC.",CT,06830,NaN,Web,2024-12-26T20:40:41.000Z,Closed with non-monetary relief,Yes,11285148.0,292,Language.ENGLISH,37,Late Payment Dispute Investigation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35988,2025-06-18T21:40:05.000Z,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,Experian Credit Services [PROTECTED] [PROTECTE...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NY,105XX,NaN,Web,2025-06-18T21:42:38.000Z,Closed with explanation,Yes,14147519.0,138,Language.ENGLISH,72,Credit File Fraud Blocking
35989,2023-11-07T18:18:13.000Z,Credit reporting or other personal consumer re...,Credit reporting,Credit monitoring or identity theft protection...,Problem canceling credit monitoring or identif...,i got a copy of my credit repor not to long a ...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,TX,77016,NaN,Web,2023-11-07T18:36:29.000Z,Closed with explanation,Yes,7822102.0,69,Language.ENGLISH,80,Unauthorized Account Reporting Disputes
35990,2024-06-18T21:31:06.000Z,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Someone opened invalid accounts using my infor...,NaN,CAPITAL ONE FINANCIAL CORPORATION,NV,89122,NaN,Web,2024-06-18T21:31:07.000Z,Closed with explanation,Yes,9287994.0,7,Language.ENGLISH,0,Identity Theft Fraud Reporting
35991,2025-10-09T18:56:15.000Z,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,"Managing, opening, or closing your mobile wall...",NaN,[PROTECTED] closed my account after I sent dri...,NaN,"Paypal Holdings, Inc",TX,77099,NaN,Web,2025-10-09T19:05:37.000Z,Closed with explanation,Yes,16471267.0,9,Language.ENGLISH,-1,Outliers / Unclassified


In [6]:
df["Topic_Label"]

0                  Formal Debt Validation Requests
1        Unauthorized Digital Payment Transactions
2                          Outliers / Unclassified
3                 Credit Card Transaction Disputes
4               Late Payment Dispute Investigation
                           ...                    
35988                   Credit File Fraud Blocking
35989      Unauthorized Account Reporting Disputes
35990               Identity Theft Fraud Reporting
35991                      Outliers / Unclassified
35992                      Outliers / Unclassified
Name: Topic_Label, Length: 35993, dtype: str

In [7]:
df[["Topic_Label"]].to_csv("labels.csv", index=False)

In [9]:
import pandas as pd


labels = (
    df[["Topic", "Topic_Label"]]
    .drop_duplicates()
    .sort_values("Topic")
)

labels.to_csv("labels.csv", index=False)

In [1]:
import pandas as pd

df = pd.read_csv("cfpb_final_with_labeled_topics.csv")

topic_lookup = (
    df[["Topic", "Topic_Label"]]
    .drop_duplicates()
    .sort_values("Topic")
)

topic_lookup.to_csv("topic_lookup.csv", index=False)

print(topic_lookup.shape)

(214, 2)
